In [ ]:
# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend
# Task 4.1 — One validated call

import json
import time
import pandas as pd
import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError


# Setup client & model & Securely ask the user for their API key using getpass
api_key = getpass.getpass("Enter your Gemini API Key: ")
client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash"

# Allowed Root Causes
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# JSON Response Schema -- response_mime_type set to JSON
# SCHEMA = {   # same idea as the name / age / city example in class: properties, their types, and required
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

# system instruction
SYSTEM_TEXT = (
    "You are a precise healthcare appointment follow-up analyst. "
    "Analyze patient follow-up notes and return only valid JSON matching the required schema."
)

def analyse_note(note_text: str) -> dict:

# 4. Prompt built with CRAFT parts (Context, Role, Action, Tone, Format)
    prompt = f"""
Context: You are reviewing patient appointment follow-up notes to classify why an appointment was missed or requires follow-up for automated triage and validation in Task 4.2.
Role: Act as a meticulous healthcare operations analyst.
Action: Read the follow-up note below and determine:
  1. root_cause: Choose ONLY ONE value from this exact list (copy spelling and capitalization verbatim): {ALLOWED_ROOT_CAUSES}
  2. confidence: Provide a confidence score between 0.0 and 1.0.
  3. summary: Write a concise summary of the note in at most 25 words.
  4. recommended_action: State a clear, practical next step for the care team.
Tone: Objective, concise, and factual.
Format: Return a single JSON object with keys root_cause, confidence, summary, and recommended_action.

Note:
\"\"\"{note_text}\"\"\"
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )

    return json.loads(response.text)

# Run on note N001 and print the parsed dictionary
# Load follow-up notes CSV specifically for this task
df_notes = pd.read_csv("/content/FollowUps.csv")

N001_text = df_notes.loc[df_notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(N001_text)
print(result)

In [ ]:
# Task 4.2 — Validate and process the 10 notes

# (a) Define a Pydantic model FollowupInsight(BaseModel) with root_cause: str = Field(min_length=3), confidence: float = Field(ge=0, le=1), summary: str = Field(min_length=5), recommended_action: str = Field(min_length=5), and write a check that root_cause is in the allowed list.

from pydantic import BaseModel, Field, field_validator

ALLOWED_ROOT_CAUSES = {
    "scheduling_conflict",
    "forgot_appointment",
    "transportation_issue",
    "weather",
    "illness",
    "other",
}


class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def validate_root_cause(cls, value: str) -> str:
        normalized = value.strip().lower()
        if normalized not in ALLOWED_ROOT_CAUSES:
            raise ValueError(
                f"root_cause must be one of {sorted(ALLOWED_ROOT_CAUSES)}, got '{value}'"
            )
        return normalized


In [ ]:
# (b) Read followup_notes.csv with pandas and loop over the first 10 rows.

import pandas as pd
import json
from pydantic import ValidationError

# Input validation: pandas reads a blank note as NaN (a float, which has no .strip()), so test pd.isna(note_text) first, then str(note_text).strip() == "".

df_notes = pd.read_csv("/content/FollowUps.csv")

note_col = "note_text" if "note_text" in df_notes.columns else "note"

results = []

# Corrected to iterate over df_notes instead of df
for _, row in df_notes.head(10).iterrows():
    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row[note_col]

    # 1. Input validation: check pd.isna first, then blank/whitespace-only strings
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

     # 2. Call analyse_note and validate with FollowupInsight + ALLOWED_ROOT_CAUSES
    raw_response = ""
    data = {}
    validation_status = "invalid"

    try:
        data = analyse_note(str(note_text))
        raw_response = json.dumps(data)

        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_ROOT_CAUSES:
            validation_status = "valid"
        else:
            validation_status = "invalid"

    except (ValidationError, json.JSONDecodeError, Exception):
        validation_status = "invalid"

    # 3. Record reply without repairing; copy existing fields and leave missing ones empty
    results.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": data.get("root_cause", "") if isinstance(data, dict) else "",
        "confidence": data.get("confidence", "") if isinstance(data, dict) else "",
        "summary": data.get("summary", "") if isinstance(data, dict) else "",
        "recommended_action": data.get("recommended_action", "") if isinstance(data, dict) else "",
        "validation_status": validation_status,
        "raw_response": raw_response,
    })


In [ ]:
# 4.2 (c) Pause between calls (import time then time.sleep(13)) to respect the free-tier rate limit.
import time
time.sleep(13)


In [ ]:
# 4.2 (d). Save to notes_analysed.csv
df_notes = pd.read_csv("/content/FollowUps.csv")

output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response",
]


notes_analysed = pd.DataFrame(results, columns=output_columns)
notes_analysed.to_csv("notes_analysed.csv", index=False)
print(notes_analysed)


In [ ]:
#Task 4.3 — Break it on purpose
# 4.3 (a) Feed the Pydantic model two deliberately bad replies and catch each with try/except ValidationError (from pydantic import ValidationError): one with confidence = 1.7, one missing recommended_action; print what each error says.

from pydantic import BaseModel, Field, ValidationError, field_validator

ALLOWED_ROOT_CAUSES = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
}


class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def validate_root_cause(cls, value: str) -> str:
        normalized = value.strip().lower()
        if normalized not in ALLOWED_ROOT_CAUSES:
            raise ValueError(
                f"root_cause must be one of {sorted(ALLOWED_ROOT_CAUSES)}, got '{value}'"
            )
        return normalized


# 1. Deliberately bad reply: confidence = 1.7 (exceeds le=1)
bad_confidence_reply = {
    "root_cause": "forgot_appointment",
    "confidence": 1.7,
    "summary": "Patient stated they completely forgot about the morning slot.",
    "recommended_action": "Send an automated SMS reminder 24 hours before the next visit.",
}

try:
    FollowupInsight(**bad_confidence_reply)
except ValidationError as e:
    print("=== Error 1: confidence = 1.7 ===")
    print(e)
    print()

# 2. Deliberately bad reply: missing recommended_action
missing_action_reply = {
    "root_cause": "transportation_issue",
    "confidence": 0.85,
    "summary": "Patient missed the bus and could not arrange a ride in time.",
}

try:
    FollowupInsight(**missing_action_reply)
except ValidationError as e:
    print("=== Error 2: missing recommended_action ===")
    print(e)


In [ ]:
# 4.3 (b) For notes N007 (Hinglish) and N009 (ambiguous), compare two prompts at temperature=0: V1 = just "Classify this note", V2 = your CRAFT prompt with the allowed values and the instruction to answer other rather than guess.

import json
import pandas as pd
from google.genai import types

#  allowed root cause list
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# Same schema used across V1 and V2 (root_cause as a plain string)
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

def run_prompt_variant(prompt_text: str) -> dict:
    """Helper to call Gemini with identical settings for V1 and V2. Safely catches quota/API exceptions."""
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_TEXT,
                temperature=0,
                max_output_tokens=1000,
                response_mime_type="application/json",
                response_schema=SCHEMA,
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
            ),
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"API Call Skipped or Quota Exceeded. Error details: {e}")
        # Return a fallback matching the schema to allow the code to run to completion
        return {
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "API quota limit reached",
            "recommended_action": "Retry later after quota resets"
        }




# Task 4.3 (b) — Compare V1 vs V2 on N007 and N009

df_notes = pd.read_csv("/content/FollowUps.csv")
target_notes = {
    "N007": df_notes.loc[df_notes["note_id"] == "N007", "note_text"].iloc[0],
    "N009": df_notes.loc[df_notes["note_id"] == "N009", "note_text"].iloc[0],
}

comparison_rows = []
calls_4_3 = 0

for note_id, note_text in target_notes.items():
    # V1: Minimal prompt
    v1_prompt = f'Classify this note:\n"""{note_text}"""'
    v1_res = run_prompt_variant(v1_prompt)
    calls_4_3 += 1
    v1_rc = v1_res.get("root_cause", "")

    comparison_rows.append({
        "Note": note_id,
        "Prompt": "V1 (Minimal)",
        "root_cause": v1_rc,
        "In Allowed List?": v1_rc in ALLOWED_ROOT_CAUSES,
    })

    # V2: CRAFT prompt with allowed list + "other rather than guess"
    v2_prompt = f"""
Context: Incoming notes contain reasons for missed appointments and must be sorted into fixed categories for routing. If a note does not clearly fit a category, label it as "other" rather than guessing.
Role: Act as a strict healthcare operations analyst and note-classification engine.
Action: Read the follow-up note below and determine:
  1. root_cause: Choose ONLY ONE value from this exact list (copy spelling and capitalization verbatim): {ALLOWED_ROOT_CAUSES}. If the note is ambiguous or does not clearly fit, output "other" rather than guessing.
  2. confidence: Provide a confidence score between 0.0 and 1.0.
  3. summary: Write a concise summary of the note in at most 25 words.
  4. recommended_action: State a clear, practical next step for the care team.
Tone: High precision, conservative, objective, and non-speculative.
Format: Return a single JSON object with keys root_cause, confidence, summary, and recommended_action.

Note:
\"\"\"{note_text}\"\"\"
"""
    v2_res = run_prompt_variant(v2_prompt)
    calls_4_3 += 1
    v2_rc = v2_res.get("root_cause", "")

    comparison_rows.append({
        "Note": note_id,
        "Prompt": "V2 (CRAFT)",
        "root_cause": v2_rc,
        "In Allowed List?": v2_rc in ALLOWED_ROOT_CAUSES,
    })

df_comparison = pd.DataFrame(comparison_rows)
print("=== Task 4.3 (b): V1 vs V2 Comparison ===")
print(df_comparison.to_markdown(index=False))


In [ ]:
# 4.3 (c) Count and print total API calls (4.1–4.3)

calls_4_1 = 1  # 1 call on N001

# In 4.2, count non-blank rows processed in the first 10 notes (typically 9, as 1 row is blank/NaN)
first_10 = df_notes.head(10)
note_col = "note_text" if "note_text" in first_10.columns else "note"
calls_4_2 = int(first_10[note_col].apply(lambda x: not pd.isna(x) and str(x).strip() != "").sum())

total_calls = calls_4_1 + calls_4_2 + calls_4_3

print("\n=== Task 4.3 (c): API Call Summary ===")
print(f"Task 4.1 API calls: {calls_4_1}")
print(f"Task 4.2 API calls: {calls_4_2} (10 rows minus blank/skipped rows)")
print(f"Task 4.3 API calls: {calls_4_3} (2 notes x 2 prompts)")
print(f"Total API calls across Tasks 4.1–4.3: {total_calls}")

In [ ]:
# Task 4.4 Merge notes_analysed.csv with appointments_clean.csv on appointment_id and cross-tabulate root_cause against reminder_sent (pd.crosstab).

import pandas as pd

# 1. Load the analysed notes and clean appointments datasets
df_notes = pd.read_csv("/content/notes_analysed.csv")
df_appts = pd.read_csv("/content/appointments_clean.csv")

# 2. Merge on appointment_id (inner join keeps only the analysed appointments)
df_merged = pd.merge(df_notes, df_appts, on="appointment_id", how="inner")
print(df_merged)

# 3. Cross-tabulate root_cause against reminder_sent (margins=True adds row/column totals)
ct = pd.crosstab(
    df_merged["root_cause"],
    df_merged["reminder_sent"],
    margins=True,
    margins_name="Total"
)

print("=== Cross-Tabulation: root_cause vs. reminder_sent ===")
print(ct)